# EECE 5644 Project – Baseline: Color Histogram + Logistic Regression
**Team:** Jingqi Mei & Andrew Chen

**Goal.** Establish the simplest reasonable result that every later model must beat, and check how hard the
six-class task really is.

**Protocol (same as the final models will use):**
- Only the 18 **training clips** are used. The 4 held-out test sequences are *not touched* here; they are reserved for the final evaluation.
- **Grouped cross-validation by clip**: no clip is ever split between training and validation.
- Every training sample gets an **out-of-fold (OOF) prediction** from a model that never saw its clip; metrics are computed on the pooled OOF predictions, plus per-fold macro-F1 (mean ± std).

Requires `scikit-learn` (`python -m pip install scikit-learn`).

In [ ]:
import os, glob, re, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import cv2
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (f1_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

In [ ]:
# ---------------- Configuration ----------------
ROOT = "./data"
ANN  = os.path.join(ROOT, "Annotations", "Annotations")
CLASSES  = ["go", "goLeft", "stop", "stopLeft", "warning", "warningLeft"]
COLOR_OF = {"go": "green", "goLeft": "green", "stop": "red", "stopLeft": "red",
            "warning": "yellow", "warningLeft": "yellow"}

K_SUBSAMPLE = 5        # keep every k-th frame of each clip (update from EDA Section 5)
MIN_H       = 5        # drop boxes shorter than this many pixels (update from EDA Section 4)
PATCH_WH    = (16, 32) # resized patch (width, height)   (update from EDA Section 4)
N_FOLDS     = 6
SEED        = 0
CACHE_DIR   = "./cache"   # add cache/ to .gitignore
os.makedirs(CACHE_DIR, exist_ok=True)

## 1. Load training annotations, filter, subsample

In [ ]:
rows = []
for f in sorted(glob.glob(f"{ANN}/**/frameAnnotationsBOX.csv", recursive=True)):
    clip = os.path.relpath(os.path.dirname(f), ANN).replace(os.sep, "/")
    if "Train" not in clip:            # skip the 4 held-out test sequences
        continue
    df = pd.read_csv(f, sep=";")
    df["clip"], df["isNight"] = clip, clip.lower().startswith("night")
    rows.append(df)
ann = pd.concat(rows, ignore_index=True).rename(columns={
    "Annotation tag": "tag", "Upper left corner X": "x1", "Upper left corner Y": "y1",
    "Lower right corner X": "x2", "Lower right corner Y": "y2"})
ann["frame"] = ann["Filename"].map(os.path.basename)
ann["frame_no"] = ann["frame"].str.extract(r"--(\d+)\.jpg$", expand=False).astype(int)
ann["h"], ann["w"] = ann.y2 - ann.y1, ann.x2 - ann.x1
n0 = len(ann)

ann = ann[ann.tag.isin(CLASSES)]                       # drops e.g. goForward
ann = ann[(ann.w > 0) & (ann.h >= MIN_H)]
ann = ann.drop_duplicates(["frame", "tag", "x1", "y1", "x2", "y2"])
ann = ann[ann.frame_no % K_SUBSAMPLE == 0].reset_index(drop=True)
print(f"{n0} boxes -> {len(ann)} after filtering and subsampling (k={K_SUBSAMPLE})")
print(ann.groupby(["isNight", "tag"]).size().unstack(fill_value=0))

## 2. Color-histogram features
For each box: crop tightly, resize to `PATCH_WH`, convert to HSV, then compute
- an 18-bin **hue histogram of lit pixels** (saturation ≥ 80 and value ≥ 120), normalised by the number of lit pixels;
- 8-bin **saturation** and **value** histograms over all pixels;
- the **fraction of lit pixels**.

That is 35 features. This is deliberately colour-only: it contains no shape information, so it should be weak at
arrow vs. circle – which is exactly what the error decomposition below will measure.

Each image is read **once** (boxes are grouped by frame) and the features are cached to disk.

In [ ]:
def color_features(rgb_patch):
    p = cv2.resize(rgb_patch, PATCH_WH, interpolation=cv2.INTER_AREA)
    hsv = cv2.cvtColor(p, cv2.COLOR_RGB2HSV)
    h, s, v = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    lit = (s >= 80) & (v >= 120)
    hue = np.histogram(h[lit], bins=18, range=(0, 180))[0] / max(lit.sum(), 1)
    sat = np.histogram(s, bins=8, range=(0, 256))[0] / s.size
    val = np.histogram(v, bins=8, range=(0, 256))[0] / v.size
    return np.concatenate([hue, sat, val, [lit.mean()]]).astype(np.float32)

cache = os.path.join(CACHE_DIR, f"baseline_color_k{K_SUBSAMPLE}_h{MIN_H}_{PATCH_WH[0]}x{PATCH_WH[1]}.npz")
if os.path.exists(cache):
    X = np.load(cache)["X"]; print("Loaded cached features:", X.shape)
else:
    FRAME_PATH = {os.path.basename(p): p
                  for p in glob.glob(f"{ROOT}/**/frames/*.jpg", recursive=True) if "sample-" not in p}
    X = np.zeros((len(ann), 35), np.float32)
    t0, by_frame = time.time(), ann.groupby("frame").indices
    for i, (frame, idx) in enumerate(by_frame.items()):
        img = cv2.cvtColor(cv2.imread(FRAME_PATH[frame]), cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]
        for j in idx:
            r = ann.iloc[j]
            x1, y1 = max(int(r.x1), 0), max(int(r.y1), 0)
            x2, y2 = min(int(r.x2), W), min(int(r.y2), H)
            X[j] = color_features(img[y1:y2, x1:x2])
        if (i + 1) % 500 == 0:
            print(f"{i + 1}/{len(by_frame)} frames, {time.time() - t0:.0f}s")
    np.savez(cache, X=X); print("Saved", cache, X.shape)

y      = ann["tag"].to_numpy()
groups = ann["clip"].to_numpy()          # note: ann.clip would be the DataFrame.clip() method

## 3. Grouped cross-validation (by clip)

In [ ]:
cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof = np.empty(len(y), dtype=object)
fold_f1 = []
for k, (tr, va) in enumerate(cv.split(X, y, groups)):
    model = make_pipeline(StandardScaler(),
                          LogisticRegression(max_iter=3000, class_weight="balanced"))
    model.fit(X[tr], y[tr])
    oof[va] = model.predict(X[va])
    f1 = f1_score(y[va], oof[va], labels=CLASSES, average="macro", zero_division=0)
    fold_f1.append(f1)
    print(f"Fold {k}: {len(set(groups[va]))} val clips, missing classes in val: "
          f"{sorted(set(CLASSES) - set(y[va])) or 'none'}, macro-F1 = {f1:.3f}")
print(f"\nPer-fold macro-F1: {np.mean(fold_f1):.3f} ± {np.std(fold_f1):.3f}")

## 4. Results

In [ ]:
print(f"Pooled OOF macro-F1:        {f1_score(y, oof, labels=CLASSES, average='macro', zero_division=0):.3f}")
print(f"Pooled OOF balanced acc.:   {balanced_accuracy_score(y, oof):.3f}\n")
print(classification_report(y, oof, labels=CLASSES, zero_division=0, digits=3))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix(y, oof, labels=CLASSES, normalize="true"),
                       display_labels=CLASSES).plot(ax=ax, values_format=".2f", colorbar=False)
ax.set_title("Baseline – row-normalised confusion matrix (OOF)")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

In [ ]:
def error_decomposition(y_true, y_pred):
    tc = np.array([COLOR_OF[t] for t in y_true]); pc = np.array([COLOR_OF[p] for p in y_pred])
    color_err = np.mean(tc != pc)                                # wrong colour
    shape_err = np.mean((tc == pc) & (y_true != y_pred))         # right colour, wrong arrow/circle
    stop_as_go = np.mean(pc[tc == "red"] == "green") if (tc == "red").any() else np.nan
    return {"6-class error": np.mean(y_true != y_pred), "color error": color_err,
            "shape error": shape_err, "3-class color acc.": 1 - color_err,
            "red -> green rate": stop_as_go}

res = {"all": error_decomposition(y, oof)}
for night in [False, True]:
    m = ann.isNight.to_numpy() == night
    if m.any():
        res["night" if night else "day"] = error_decomposition(y[m], oof[m])
pd.DataFrame(res).round(4)

## 5. How to read the results
- **Color error small, shape error large** → colour features already solve the colour part; the bottleneck is arrow vs. circle, so HOG (shape) features should give the biggest gain.
- **3-class colour accuracy near 100%** → confirms the 3-class task is nearly saturated and justifies the six-class focus.
- **Night clearly worse than day** → supports night-specific analysis and GMM class-conditionals.
- **Large spread across folds** → results depend strongly on which clips are held out; report mean ± std, not a single number.
- Record the pooled macro-F1 as *the number every later model must beat*.